In [1]:
"""
Pharmacy Operations & Demand Intelligence — Synthetic Demonstration
06_python/01_data_generation.py

Generates 6 CSVs matching 05_sql/01_create_tables_mysql.sql:
  Suppliers, Product_Master, Sales, Stock_Daily, Orders, Customer_Requests

Key design choices (per project spec, Section 5):
- SALES != DEMAND: Customer_Requests is generated as an independent signal,
  not derived mechanically from stockouts.
- Logging discipline is imperfect: only a fraction of real stockout events
  become a logged Customer_Request (mirrors "do you actually record
  requests?" validation question).
- One category ("Antimalarial") deliberately mirrors the real pharmacist
  conversation: 1 routinely-stocked option + 2 order-on-request alternatives.
- Noise included: supplier delays, partial receipts, unresolved/open orders,
  occasional stock adjustments, inactive supplier/product edge cases for QC.
"""

import os
import numpy as np
import pandas as pd
from datetime import date, timedelta

RNG = np.random.default_rng(42)
START_DATE = date(2025, 1, 1)
END_DATE = date(2025, 9, 30)
DATES = pd.date_range(START_DATE, END_DATE, freq="D")
N_DAYS = len(DATES)
OUT_DIR = "/home/claude/data"
os.makedirs(OUT_DIR, exist_ok=True)

# =====================================================================
# 1. SUPPLIERS
# =====================================================================
suppliers = pd.DataFrame({
    "Supplier_ID": range(1, 7),
    "Supplier_Name": [
        "Central Wholesale Distributors Ltd", "Meridian Pharma Supplies",
        "Nairobi Regional Pharma Distributors", "Baraka Health Products Ltd",
        "Highland Medical Supplies", "Coastline Pharma Traders",
    ],
    "Supplier_Category": [
        "Wholesale Pharmaceutical", "Wholesale Pharmaceutical", "General Distributor",
        "Wholesale Pharmaceutical", "General Distributor", "Wholesale Pharmaceutical",
    ],
    "Typical_Lead_Time_Days": [2, 3, 5, 3, 4, 6],
    "Active": [1, 1, 1, 1, 1, 0],
})

# =====================================================================
# 2. PRODUCT_MASTER
# =====================================================================
categories = {
    "Antimalarial": 3, "Analgesic": 8, "Antibiotic": 9, "Antihistamine": 6,
    "Cough & Cold": 8, "Gastro/Antacid": 7, "Vitamins & Supplements": 9,
    "Dermatological": 7, "Antiseptic/First Aid": 6, "Family Planning": 7,
}
forms = ["Tablet", "Syrup", "Cream", "Capsule", "Injection", "Suspension", "Ointment"]
form_p = [0.42, 0.14, 0.12, 0.14, 0.03, 0.10, 0.05]

rows, pid = [], 101
active_suppliers = suppliers.loc[suppliers.Active == 1, "Supplier_ID"].tolist()

for cat, n in categories.items():
    for i in range(n):
        letter = chr(65 + i)
        form = RNG.choice(forms, p=form_p)
        unit_cost = round(float(RNG.uniform(15, 250)), 2)
        selling_price = round(unit_cost * float(RNG.uniform(1.25, 1.7)), 2)
        supplier_id = int(RNG.choice(active_suppliers))

        if cat == "Antimalarial":
            routinely_stocked = 1 if letter == "A" else 0
        else:
            routinely_stocked = 1 if RNG.random() < 0.72 else 0
        active = 1 if RNG.random() > 0.03 else 0

        rows.append({
            "Product_ID": pid, "Product_Name": f"{cat} - Option {letter}", "Category": cat,
            "Subcategory": f"{cat} type {letter}" if RNG.random() < 0.4 else None,
            "Dosage_Form": form,
            "Strength": f"{int(RNG.choice([20,40,100,125,250,500]))}mg" if RNG.random() < 0.6 else None,
            "Pack_Size": f"{int(RNG.choice([10,12,20,24,30,50,100]))} units",
            "Supplier_ID": supplier_id, "Unit_Cost": unit_cost, "Selling_Price": selling_price,
            "Routinely_Stocked": routinely_stocked, "Active": active,
        })
        pid += 1

product_master = pd.DataFrame(rows)
tiers = np.where(
    (product_master.Routinely_Stocked == 1) & (product_master.Active == 1),
    RNG.choice(["Fast", "Medium", "Slow"], size=len(product_master), p=[0.28, 0.47, 0.25]),
    "N/A",
)
tier_lookup = dict(zip(product_master.Product_ID, tiers))
BASE_RATE = {"Fast": 1.3, "Medium": 0.55, "Slow": 0.15, "N/A": 0.0}
SEASONAL = {
    "Antimalarial": (range(20, 31), 1.9),
    "Antihistamine": (range(6, 13), 1.5),
    "Cough & Cold": (range(26, 35), 1.4),
}

def season_mult(cat, week):
    if cat in SEASONAL:
        weeks, boost = SEASONAL[cat]
        if week in weeks:
            return boost
    return 1.0

# =====================================================================
# 3 & 4. SALES + STOCK_DAILY + ROUTINE/STOCK-SHORTAGE ORDERS (stocked products)
# =====================================================================
sales_rows, stock_rows, orders_rows = [], [], []
sale_id, stock_id, order_id = 500001, 1, 9001

state = {}
for _, p in product_master.iterrows():
    pid_ = p.Product_ID
    if p.Routinely_Stocked == 1 and p.Active == 1:
        tier = tier_lookup[pid_]
        opening = int({"Fast": 60, "Medium": 30, "Slow": 15}[tier] * RNG.uniform(0.85, 1.15))
        state[pid_] = {"stock": opening, "reorder_point": int(opening * 0.30),
                        "target": opening, "due_date": None, "due_qty": 0}
    else:
        state[pid_] = {"stock": 0, "reorder_point": 0, "target": 0, "due_date": None, "due_qty": 0}

# store stockout events (product, date, unmet_units) to sample logged requests from later
stockout_events = []

active_products = product_master.loc[product_master.Active == 1]

for d in DATES:
    d_date = d.date()
    week = int(d.isocalendar()[1])
    weekend = d.weekday() >= 5

    for _, p in active_products.iterrows():
        pid_ = p.Product_ID
        st = state[pid_]

        # capture TRUE opening stock (before today's receipt is applied)
        opening_stock = st["stock"]

        # receive due order (added to available stock for today, tracked separately)
        received_today = 0
        if st["due_date"] == d_date:
            received_today = st["due_qty"]
            st["due_date"], st["due_qty"] = None, 0

        if p.Routinely_Stocked != 1:
            stock_rows.append({"Stock_ID": stock_id, "Date": d_date, "Product_ID": pid_,
                                "Opening_Stock": 0, "Received": 0, "Sold": 0,
                                "Adjustments": 0, "Closing_Stock": 0})
            stock_id += 1
            continue

        available_stock = opening_stock + received_today
        tier = tier_lookup[pid_]
        rate = BASE_RATE[tier] * season_mult(p.Category, week) * (0.85 if weekend else 1.0)
        demand_units = int(RNG.poisson(max(rate, 0.01)))
        sold_units = min(demand_units, available_stock)
        unmet_units = demand_units - sold_units

        if sold_units > 0:
            price, cost = float(p.Selling_Price), float(p.Unit_Cost)
            sales_rows.append({"Sale_ID": sale_id, "Sale_Date": d_date, "Product_ID": pid_,
                                "Quantity": int(sold_units), "Unit_Selling_Price": price,
                                "Revenue": round(price * sold_units, 2), "Unit_Cost": cost,
                                "Margin": round((price - cost) * sold_units, 2)})
            sale_id += 1

        adjustment = -int(RNG.integers(1, 4)) if RNG.random() < 0.01 else 0
        closing_stock = max(available_stock - sold_units + adjustment, 0)

        stock_rows.append({"Stock_ID": stock_id, "Date": d_date, "Product_ID": pid_,
                            "Opening_Stock": int(opening_stock), "Received": int(received_today),
                            "Sold": int(sold_units), "Adjustments": int(adjustment),
                            "Closing_Stock": int(closing_stock)})
        stock_id += 1
        st["stock"] = closing_stock

        if unmet_units > 0:
            stockout_events.append((pid_, d_date, unmet_units))

        # reorder logic
        if st["due_date"] is None and closing_stock <= st["reorder_point"]:
            sup = suppliers.loc[suppliers.Supplier_ID == p.Supplier_ID].iloc[0]
            lead = int(sup.Typical_Lead_Time_Days)
            delay = int(RNG.integers(3, 9)) if RNG.random() < 0.12 else 0
            actual_lead = lead + delay
            reason = "Routine Replenishment" if closing_stock > 0 else "Stock Shortage"
            qty_needed = max(st["target"] - closing_stock, 1)
            order_date = d_date
            expected_date = order_date + timedelta(days=lead)
            unresolved = RNG.random() < 0.04
            received_date = None if unresolved else order_date + timedelta(days=actual_lead)
            qty_received = None
            if received_date is not None and received_date <= END_DATE:
                qty_received = qty_needed if RNG.random() > 0.10 else int(qty_needed * RNG.uniform(0.7, 0.95))
                st["due_date"], st["due_qty"] = received_date, qty_received
            elif received_date is not None:
                # would arrive after our data window closes -> leave unresolved in this dataset
                received_date, qty_received = None, None

            orders_rows.append({"Order_ID": order_id, "Order_Date": order_date, "Product_ID": pid_,
                                 "Quantity_Ordered": int(qty_needed), "Supplier_ID": int(p.Supplier_ID),
                                 "Reason": reason, "Expected_Date": expected_date,
                                 "Received_Date": received_date, "Quantity_Received": qty_received})
            order_id += 1

print(f"Stocked-product pass done. Sales={len(sales_rows)} Stock={len(stock_rows)} Orders={len(orders_rows)}")

# =====================================================================
# 5. CUSTOMER_REQUESTS (independent demand signal) + request-triggered Orders/Sales
# =====================================================================
requests_rows = []
request_id = 70001

# --- 5a. Requests for NON-routinely-stocked, active products (the core scenario) ---
non_stocked = product_master.loc[(product_master.Routinely_Stocked == 0) & (product_master.Active == 1)]
popularity = {pid_: float(RNG.uniform(0.02, 0.30)) for pid_ in non_stocked.Product_ID}
# give the two Antimalarial alternatives a deliberate, realistic bump (mirrors real conversation)
for pid_, cat in zip(non_stocked.Product_ID, non_stocked.Category):
    if cat == "Antimalarial":
        popularity[pid_] = float(RNG.uniform(0.18, 0.30))

alt_pool = {}  # category -> list of product ids in that category (any stock status) for plausible alternatives
for cat in categories:
    alt_pool[cat] = product_master.loc[(product_master.Category == cat) & (product_master.Active == 1), "Product_ID"].tolist()

for pid_, cat in zip(non_stocked.Product_ID, non_stocked.Category):
    base_rate = popularity[pid_]
    for d in DATES:
        d_date = d.date()
        week = int(d.isocalendar()[1])
        rate = base_rate * season_mult(cat, week)
        if RNG.random() >= rate:
            continue  # no request that day for this product

        qty_req = int(RNG.integers(1, 3))
        category_only = RNG.random() < 0.15  # sometimes logged only as a category ask
        alt_offered = RNG.random() < 0.65
        alt_product = None
        outcome, ordered_for_customer, order_id_link, fulfilled_date = None, 0, None, None

        if alt_offered:
            candidates = [x for x in alt_pool[cat] if x != pid_]
            if candidates:
                alt_product = int(RNG.choice(candidates))

        will_order = RNG.random() < 0.45  # of unavailable requests, ~45% result in a special order
        if will_order:
            sup = product_master.loc[product_master.Product_ID == pid_, "Supplier_ID"].iloc[0]
            sup_row = suppliers.loc[suppliers.Supplier_ID == sup].iloc[0]
            lead = int(sup_row.Typical_Lead_Time_Days) + (int(RNG.integers(2, 7)) if RNG.random() < 0.15 else 0)
            order_date = d_date
            received_date = order_date + timedelta(days=lead)
            resolved = received_date <= END_DATE and RNG.random() > 0.05
            qty_received = qty_req if resolved and RNG.random() > 0.1 else (
                int(qty_req * RNG.uniform(0.5, 1.0)) if resolved else None)

            orders_rows.append({
                "Order_ID": order_id, "Order_Date": order_date, "Product_ID": pid_,
                "Quantity_Ordered": qty_req, "Supplier_ID": int(sup),
                "Reason": "Customer Request", "Expected_Date": order_date + timedelta(days=int(sup_row.Typical_Lead_Time_Days)),
                "Received_Date": received_date if resolved else None,
                "Quantity_Received": qty_received,
            })
            order_id_link = order_id
            order_id += 1

            if resolved:
                fulfilled_date = received_date
                outcome = "Ordered and Fulfilled"
                ordered_for_customer = 1
                price = float(product_master.loc[product_master.Product_ID == pid_, "Selling_Price"].iloc[0])
                cost = float(product_master.loc[product_master.Product_ID == pid_, "Unit_Cost"].iloc[0])
                sold_qty = min(qty_received or qty_req, qty_req)
                if sold_qty > 0:
                    sales_rows.append({"Sale_ID": sale_id, "Sale_Date": fulfilled_date, "Product_ID": pid_,
                                        "Quantity": int(sold_qty), "Unit_Selling_Price": price,
                                        "Revenue": round(price * sold_qty, 2), "Unit_Cost": cost,
                                        "Margin": round((price - cost) * sold_qty, 2)})
                    sale_id += 1
            else:
                outcome = "Other"  # order placed but not yet resolved within data window
                ordered_for_customer = 1
        elif alt_offered and RNG.random() < 0.6:
            outcome = "Alternative Accepted"
        elif RNG.random() < 0.5:
            outcome = "Customer Declined"
        else:
            outcome = "Unavailable"

        requests_rows.append({
            "Request_ID": request_id, "Request_Date": d_date,
            "Requested_Product_ID": None if category_only else int(pid_),
            "Requested_Category": cat if category_only else None,
            "Quantity_Requested": qty_req, "Available": 0,
            "Alternative_Offered": int(alt_offered), "Alternative_Product_ID": alt_product,
            "Ordered_For_Customer": ordered_for_customer, "Order_ID": order_id_link,
            "Fulfilled_Date": fulfilled_date, "Outcome": outcome,
        })
        request_id += 1

# --- 5b. Logged requests arising from real stockouts of routinely-stocked products ---
# Only a fraction of true stockout events get logged (imperfect logging discipline)
for pid_, d_date, unmet in stockout_events:
    if RNG.random() >= 0.35:
        continue
    cat = product_master.loc[product_master.Product_ID == pid_, "Category"].iloc[0]
    alt_offered = RNG.random() < 0.75
    alt_product = None
    if alt_offered:
        candidates = [x for x in alt_pool[cat] if x != pid_]
        if candidates:
            alt_product = int(RNG.choice(candidates))
    if alt_offered and RNG.random() < 0.7:
        outcome, ordered_for_customer = "Alternative Accepted", 0
    elif RNG.random() < 0.5:
        outcome, ordered_for_customer = "Unavailable", 0
    else:
        outcome, ordered_for_customer = "Customer Declined", 0

    requests_rows.append({
        "Request_ID": request_id, "Request_Date": d_date, "Requested_Product_ID": int(pid_),
        "Requested_Category": None, "Quantity_Requested": int(min(unmet, 3)), "Available": 0,
        "Alternative_Offered": int(alt_offered), "Alternative_Product_ID": alt_product,
        "Ordered_For_Customer": ordered_for_customer, "Order_ID": None,
        "Fulfilled_Date": None, "Outcome": outcome,
    })
    request_id += 1

# --- 5c. A smaller set of "Available=1" logged requests (customer asked, it was in stock, sold) ---
# Mirrors the minority of cases where staff logged a request even though it was fulfilled normally.
stocked_active = product_master.loc[(product_master.Routinely_Stocked == 1) & (product_master.Active == 1)]
sample_n = int(len(stocked_active) * 6)  # sparse, occasional logging
for _ in range(sample_n):
    p = stocked_active.sample(1, random_state=int(RNG.integers(0, 1_000_000))).iloc[0]
    d_date = (START_DATE + timedelta(days=int(RNG.integers(0, N_DAYS)))).__class__ and DATES[int(RNG.integers(0, N_DAYS))].date()
    requests_rows.append({
        "Request_ID": request_id, "Request_Date": d_date, "Requested_Product_ID": int(p.Product_ID),
        "Requested_Category": None, "Quantity_Requested": int(RNG.integers(1, 3)), "Available": 1,
        "Alternative_Offered": 0, "Alternative_Product_ID": None, "Ordered_For_Customer": 0,
        "Order_ID": None, "Fulfilled_Date": d_date, "Outcome": "Sold",
    })
    request_id += 1

print(f"Requests generated: {len(requests_rows)} | Orders total: {len(orders_rows)} | Sales total: {len(sales_rows)}")

# =====================================================================
# EXPORT
# =====================================================================
suppliers.to_csv(f"{OUT_DIR}/Suppliers.csv", index=False)
product_master.to_csv(f"{OUT_DIR}/Product_Master.csv", index=False)
pd.DataFrame(sales_rows).sort_values(["Sale_Date", "Sale_ID"]).to_csv(f"{OUT_DIR}/Sales.csv", index=False)
pd.DataFrame(stock_rows).sort_values(["Date", "Product_ID"]).to_csv(f"{OUT_DIR}/Stock_Daily.csv", index=False)
pd.DataFrame(orders_rows).sort_values(["Order_Date", "Order_ID"]).to_csv(f"{OUT_DIR}/Orders.csv", index=False)
pd.DataFrame(requests_rows).sort_values(["Request_Date", "Request_ID"]).to_csv(f"{OUT_DIR}/Customer_Requests.csv", index=False)

print("\nRow counts:")
for f in ["Suppliers", "Product_Master", "Sales", "Stock_Daily", "Orders", "Customer_Requests"]:
    df = pd.read_csv(f"{OUT_DIR}/{f}.csv")
    print(f"  {f}: {len(df)} rows")

Stocked-product pass done. Sales=5821 Stock=18564 Orders=329
Requests generated: 1314 | Orders total: 768 | Sales total: 6228

Row counts:
  Suppliers: 6 rows
  Product_Master: 70 rows
  Sales: 6228 rows
  Stock_Daily: 18564 rows
  Orders: 768 rows
  Customer_Requests: 1314 rows


In [2]:
import os

# Paths relative to 06_python/, matching your actual folder structure
RAW_DIR = "../04_synthetic_data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

suppliers.to_csv(f"{RAW_DIR}/Suppliers.csv", index=False)
product_master.to_csv(f"{RAW_DIR}/Product_Master.csv", index=False)
pd.DataFrame(sales_rows).sort_values(["Sale_Date", "Sale_ID"]).to_csv(f"{RAW_DIR}/Sales.csv", index=False)
pd.DataFrame(stock_rows).sort_values(["Date", "Product_ID"]).to_csv(f"{RAW_DIR}/Stock_Daily.csv", index=False)
pd.DataFrame(orders_rows).sort_values(["Order_Date", "Order_ID"]).to_csv(f"{RAW_DIR}/Orders.csv", index=False)
pd.DataFrame(requests_rows).sort_values(["Request_Date", "Request_ID"]).to_csv(f"{RAW_DIR}/Customer_Requests.csv", index=False)

print("Saved to:", os.path.abspath(RAW_DIR))
for f in ["Suppliers", "Product_Master", "Sales", "Stock_Daily", "Orders", "Customer_Requests"]:
    print(f"  {f}.csv:", len(pd.read_csv(f"{RAW_DIR}/{f}.csv")), "rows")

Saved to: C:\Users\USER\Desktop\pharmacy-demand-intelligence\04_synthetic_data\raw
  Suppliers.csv: 6 rows
  Product_Master.csv: 70 rows
  Sales.csv: 6228 rows
  Stock_Daily.csv: 18564 rows
  Orders.csv: 768 rows
  Customer_Requests.csv: 1314 rows
